In [2]:
from google.colab import drive
import os
import pandas as pd
import numpy as np
from concurrent.futures import ProcessPoolExecutor
from tqdm import tqdm
import re

drive.mount('/content/drive')

# Directories
base_path = "/content/drive/My Drive/0. Liquidity and Market Stress/Crypto Raw Data_Professor"
trades_folder = f"{base_path}/SHIB-USD/trades"
quotes_folder = f"{base_path}/SHIB-USD/quotes"

Mounted at /content/drive


In [3]:
# Get the target file by months
def get_month_groups(folder):
    files = os.listdir(folder)
    pattern = r"\d{4}-\d{2}-\d{2}"
    month_groups = {}

    for f in files:
        match = re.search(pattern, f)
        if match:
            date_str = match.group()
            year_month = pd.to_datetime(date_str).strftime("%m_%Y")
            if year_month not in month_groups:
                month_groups[year_month] = []
            month_groups[year_month].append(date_str)
    return month_groups

In [4]:
def process_day(date_str):
    try:
        # Load the trades data
        trades_path = f"{trades_folder}/coinbase_trades_{date_str}_SHIB-USD.csv.gz"
        trades_df = pd.read_csv(trades_path, compression='gzip')
        trades_df['datetime'] = pd.to_datetime(trades_df['timestamp'], unit='us')

        # Load the quotes data
        quotes_path = f"{quotes_folder}/coinbase_quotes_{date_str}_SHIB-USD.csv.gz"
        quotes_df = pd.read_csv(quotes_path, compression='gzip')
        quotes_df['datetime'] = pd.to_datetime(quotes_df['timestamp'], unit='us')
        quotes_df['mid_price'] = (quotes_df['ask_price'] + quotes_df['bid_price']) / 2

        # Calculate spread as a fraction of mid_price
        quotes_df['spread'] = (quotes_df['ask_price'] - quotes_df['bid_price']) / quotes_df['mid_price']

        # Calculate market depth within ±1% of mid_price
        quotes_df['within_1pct_ask'] = quotes_df['ask_price'] <= quotes_df['mid_price'] * 1.01
        quotes_df['within_1pct_bid'] = quotes_df['bid_price'] >= quotes_df['mid_price'] * 0.99
        quotes_df['ask_depth'] = np.where(quotes_df['within_1pct_ask'], quotes_df['ask_amount'], 0)
        quotes_df['bid_depth'] = np.where(quotes_df['within_1pct_bid'], quotes_df['bid_amount'], 0)
        quotes_df['depth'] = quotes_df['ask_depth'] + quotes_df['bid_depth']

        # Merge trades and quotes
        merged = pd.merge_asof(
            trades_df.sort_values('datetime'),
            quotes_df[['datetime', 'mid_price', 'spread', 'depth']].sort_values('datetime'),
            on='datetime',
            direction='backward',
            tolerance=pd.Timedelta('2s')
        )

        # Calculate effective spread
        merged['effective_spread'] = 2 * abs(merged['price'] - merged['mid_price']) / merged['mid_price']

        # Resample to minute-level data
        resampled = merged.resample('min', on='datetime').agg({
            'spread': 'mean',                # Average bid-ask spread
            'depth': 'mean',                 # Average market depth
            'amount': 'sum',                 # Total traded volume
            'price': 'mean',                 # Average trade price
            'effective_spread': 'mean',      # Average effective spread
        })
        resampled['n_trades'] = merged.resample('min', on='datetime').size()  # Number of trades

        # Rename columns for clarity
        resampled.rename(columns={
            'spread': 'spread',
            'depth': 'depth',
            'amount': 'volume',
            'price': 'avg_trade_price',
            'effective_spread': 'avg_e_spread',
            'n_trades': 'n_trades'
        }, inplace=True)

        return resampled

    except Exception as e:
        print(f"Error processing {date_str}: {str(e)}")
        return pd.DataFrame()

In [5]:
def main():
    trade_months = get_month_groups(trades_folder)
    quote_months = get_month_groups(quotes_folder)
    common_months = set(trade_months.keys()) & set(quote_months.keys())

    # Convert common months to datetime format and sort
    sorted_months = sorted(common_months, key=lambda x: pd.to_datetime(x, format="%m_%Y"))

    for month in tqdm(sorted_months, desc="Processing months", position=0):
        # Get all the dates of the current month
        dates = sorted(list(set(trade_months[month]) & set(quote_months[month])))

        # Parallelly process the data of a single day
        with ProcessPoolExecutor() as executor:
            results = list(tqdm(executor.map(process_day, dates),
                                total=len(dates),
                                desc=f"Processing days in {month}",
                                position=1,
                                leave=False))

        # Merge results for the entire month
        month_df = pd.concat(results).sort_index()

        # Save to file
        formatted_month = pd.to_datetime(month, format="%m_%Y").strftime("%Y_%m")
        output_path = f"/content/drive/My Drive/0. Liquidity and Market Stress/Python Code/batch_process_minute_e_spread/5-year_minute_data_by_coins/SHIB/minute_liquidity_{formatted_month}.csv"
        month_df.to_csv(output_path, index_label='datetime')
        print(f"✅ {month} has been saved to: {output_path}")

In [6]:
if __name__ == "__main__":
    main()

Processing months:   2%|▏         | 1/41 [01:55<1:17:14, 115.86s/it]

✅ 09_2021 has been saved to: /content/drive/My Drive/0. Liquidity and Market Stress/Python Code/batch_process_minute_e_spread/5-year_minute_data_by_coins/SHIB/minute_liquidity_2021_09.csv



Processing months:   5%|▍         | 2/41 [05:00<1:41:46, 156.59s/it]

✅ 10_2021 has been saved to: /content/drive/My Drive/0. Liquidity and Market Stress/Python Code/batch_process_minute_e_spread/5-year_minute_data_by_coins/SHIB/minute_liquidity_2021_10.csv



Processing months:   7%|▋         | 3/41 [06:58<1:27:49, 138.67s/it]

...

✅ 11_2024 has been saved to: /content/drive/My Drive/0. Liquidity and Market Stress/Python Code/batch_process_minute_e_spread/5-year_minute_data_by_coins/SHIB/minute_liquidity_2024_11.csv



Processing months:  98%|█████████▊| 40/41 [41:48<01:22, 82.84s/it]

✅ 12_2024 has been saved to: /content/drive/My Drive/0. Liquidity and Market Stress/Python Code/batch_process_minute_e_spread/5-year_minute_data_by_coins/SHIB/minute_liquidity_2024_12.csv



Processing months: 100%|██████████| 41/41 [41:57<00:00, 61.41s/it]

✅ 01_2025 has been saved to: /content/drive/My Drive/0. Liquidity and Market Stress/Python Code/batch_process_minute_e_spread/5-year_minute_data_by_coins/SHIB/minute_liquidity_2025_01.csv
